# Predictive Concept Decoders (PCD) — a small-scale replication

**Paper:** [*Predictive Concept Decoders: Training Scalable End-to-End Interpretability Assistants*](https://arxiv.org/abs/2512.15712) — Huang, Choi, Johnson, Schwettmann, Steinhardt (Transluce, Dec 2025), arXiv:2512.15712.

**No official code exists** (only a web demo at [translude.org/pcd](https://transluce.org/pcd)), so everything below is re-implemented from the paper's equations, figures, and appendix. Every step cites the section/figure it implements.

| | |
|---|---|
| **Assumed GPU** | Free Colab **T4 (16 GB)** works with the default config. L4 / A100 presets included. |
| **Runs on free Colab?** | ✅ Yes (default preset: 1B subject model, ~1M-token budget) |
| **Expected runtime** | T4: ~30–50 min per training run × 2 runs + ~20 min of evals ≈ **1.5–2.5 h total**. L4/A100: faster per step, bigger budgets. |
| **Expected cost** | $0 on free Colab. Optional Claude auto-interp scoring ≈ **$1–3** of API credit (Haiku). Optional Colab Pro session ≪ $20. |
| **Installs** | `transformers`, `datasets`, `peft`, `accelerate`, `anthropic` (next code cell) |
| **Hugging Face token** | Needed for `meta-llama/*` models (accept the license on the model page, then add `HF_TOKEN` to Colab secrets). A no-token fallback (`Qwen/Qwen2.5-1.5B-Instruct`) is provided. |

**How to use this notebook.** It is split into numbered steps. Every training/eval step writes its artifacts to disk (`pcd_runs/`), so steps are independently re-runnable: after a Colab disconnect, re-run Steps 0–2 (setup, data, model — a few minutes) and then jump back to wherever you were. Cells that cost real time say so at the top.

> ⚠️ Colab's local disk is wiped when the runtime is recycled. If you want your checkpoints to survive, mount Drive and point `RUN_DIR` there (commented lines in the config cell).

## Step 0 · The paper in one page — the right answer first

### What the paper claims

**The core idea (§1, §2).** Instead of hand-designed interpretability agents, *train* an interpretability assistant end-to-end. The training signal: **predicting the subject model's behavior from its internal activations**, through a *communication bottleneck*:

- An **encoder** reads the subject model's residual-stream activations and compresses each token's activation vector into a **sparse list of $k$ concepts** (a linear map + top-$k$, like a sparse autoencoder's encoder).
- A **decoder** (a copy of the subject model + LoRA) receives *only* those re-embedded concepts — patched into its residual stream as soft tokens — plus a natural-language question, and must answer correctly.
- Crucially, the **encoder never sees the question**, so it must learn *general-purpose* concepts, and the **decoder never sees the raw activations**, so all information flows through the sparse bottleneck. Sparsity is what makes the explanation auditable by humans.

**The claims we will test at small scale:**

| # | Claim | Where in paper | Expected picture |
|---|---|---|---|
| **C1** | The decoder gets *steadily better at predicting suffix tokens from activations* as pretraining data grows | §3.1, **Fig 3 (left)** | Training/eval loss decreases smoothly with tokens; the decoder beats a "no-activations" baseline, i.e. real information flows through the bottleneck |
| **C2** | **Without the auxiliary loss**, concepts die off during training (≈⅓ dead by 72M tokens); with it, >90% stay active | §3.2, **Fig 13 (left)** | Active-concept fraction: aux run stays high, no-aux run sags |
| **C3** | Concept **precision (auto-interp score)** and **recall (attribute coverage)** improve with data *with* the aux loss, but **plateau or decline without it** | §3.3, **Fig 3 (middle, right)** | Two scaling curves per metric; the no-aux curve flattens/drops relative to the aux curve |

**What the full paper shows beyond this** (we skip these for compute — see the final cell): finetuning the decoder for QA on SynthSys (§4), SAE / KL-SAE baselines (§3.3, Fig 4), and the case studies — jailbreak detection, secret hints, implanted concepts, refusal auditing (§5). We include a mini version of the implanted-concept probe (§5.3) as an optional finale.

### How to read our results honestly

The paper trains on **Llama-3.1-8B-Instruct** with a 32,768-concept dictionary for **9M–144M tokens** on ~H100-class hardware. We train a **1B model** with a ~16,384-concept dictionary for **~1M tokens** on a free T4. So:

- ✅ We *can* test whether the **qualitative scaling behavior** reproduces: loss falling steadily as data grows (C1), concept death and its rescue by the aux loss (C2), and the direction of the precision/recall gap (C3).
- ❌ We can *not* match absolute numbers, and our token budget sits at the *left edge* of the paper's x-axis — their precision/recall divergence grows in the 36M→72M range, so at our scale expect **C2 to be the crispest signal** (concept death happens early) and the C3 gap to be present but noisy.

Keep that calibration in mind every time a plot appears below.

In [ ]:
# ── Installs (~1-2 min on Colab; torch itself is preinstalled) ────────────────
# transformers : subject/decoder models        peft    : LoRA adapters (§2)
# datasets     : FineWeb streaming (§3.1)      anthropic: optional auto-interp judge (§3.3)
%pip install -q -U transformers datasets peft accelerate anthropic

In [ ]:
# ── Step 0.1 · Environment check + configuration ─────────────────────────────
# Presets keep the paper's *ratios* (Fig 2, §3.1, A.1) while shrinking absolute
# scale to fit the GPU you actually have. Everything paper-vs-us is annotated.
import contextlib, gc, json, math, os, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def get_secret(name: str) -> str:
    """Colab secret if available, else environment variable, else ''."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

HF_TOKEN = get_secret("HF_TOKEN")  # needed for meta-llama/* (accept license first)

dev = "cuda" if torch.cuda.is_available() else "cpu"

def pick_preset():
    if dev != "cuda":
        return "CPU"
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    return "T4" if gb < 20 else ("L4" if gb < 32 else "A100")

PRESET = pick_preset()

# 📄 Paper: subject = Llama-3.1-8B-Instruct, 72M-144M token budgets, batch 128 (§3.1, A.1)
# 🔧 We:    1B subject (T4/L4) or 3B (A100), ~1M-6M tokens, batch 16-32
# 💡 Why:   two 8B forward passes + LoRA training don't fit 16 GB; we test *trends*, not absolute numbers.
PRESETS = {
    #        subject model                        token budget  micro-batch
    "T4":   ("meta-llama/Llama-3.2-1B-Instruct",   1_000_000,   16),
    "L4":   ("meta-llama/Llama-3.2-1B-Instruct",   3_000_000,   32),
    "A100": ("meta-llama/Llama-3.2-3B-Instruct",   6_000_000,   32),
    "CPU":  ("meta-llama/Llama-3.2-1B-Instruct",      20_000,    4),  # debug only — very slow
}

class CFG:
    # ---- model & scale (overridable below) ----
    model_name, token_budget, micro_batch = PRESETS[PRESET]
    # ---- segmentation (§3.1: n_prefix = n_middle = n_suffix = 16) — unchanged ----
    n_prefix, n_middle, n_suffix = 16, 16, 16
    # ---- encoder (§3.1: m = 32768 = 8× expansion of d=4096; k = 16) ----
    expansion = 8      # m = expansion * d  → same 8× ratio as the paper
    k = 16             # active concepts per token — unchanged
    # ---- decoder LoRA (A.1: alpha=32, dropout=0.05; rank swept 4-16 in Fig 15, main-run rank unstated) ----
    lora_r, lora_alpha, lora_dropout = 8, 32, 0.05
    # ---- optimization (A.1: lr=1e-4, wd=0.01, cosine schedule, no warmup) — unchanged except batch ----
    lr, weight_decay = 1e-4, 0.01
    # ---- auxiliary loss (§3.2 + A.1: eps_aux=1e-4, k_aux=500 of 32768, window = 1M of 72M tokens) ----
    eps_aux = 1e-4
    # k_aux and dead_window are set after model load, scaled to our m / budget.
    # ---- evaluation sizes (ours; paper uses 400 concepts + SynthSys) ----
    eval_examples      = 256    # held-out FineWeb windows for suffix-loss eval
    exemplar_docs      = 1536   # held-out windows for auto-interp exemplars
    autointerp_concepts = 64    # 📄 400 concepts (§3.3) → 🔧 64, to keep API cost ≈ $1-3
    seed = 0

# ── Manual overrides — uncomment to change ───────────────────────────────────
# CFG.model_name = "Qwen/Qwen2.5-1.5B-Instruct"      # ✅ ungated: no HF_TOKEN needed
# CFG.model_name = "meta-llama/Llama-3.2-3B-Instruct" # Colab Pro (L4/A100)
# CFG.model_name = "meta-llama/Llama-3.1-8B-Instruct" # A100-40GB only: also set micro_batch=4
# CFG.token_budget = 8_000_000                        # bigger scaling sweep on A100

# Milestones at budget × {1/8, 1/4, 1/2, 1} — mirrors the paper's ×2-spaced
# budgets {9M, 18M, 36M, 72M} (Fig 3), shifted way down in absolute scale.
CFG.milestones = [CFG.token_budget // 8, CFG.token_budget // 4,
                  CFG.token_budget // 2, CFG.token_budget]

RUN_DIR = Path("pcd_runs")
# To persist across Colab sessions, use Drive instead:
# from google.colab import drive; drive.mount("/content/drive")
# RUN_DIR = Path("/content/drive/MyDrive/pcd_runs")
DATA_DIR = RUN_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if dev == "cuda":
        torch.cuda.manual_seed_all(s)

set_seed(CFG.seed)
# shared plot styling for the two ablation runs (used across several steps)
COL = {"aux": "tab:blue", "noaux": "tab:orange"}
LBL = {"aux": "PCD", "noaux": "PCD (no aux loss)"}
print(f"Device: {dev} ({torch.cuda.get_device_name(0) if dev=='cuda' else 'no GPU'})")
print(f"Preset: {PRESET} → {CFG.model_name}, budget {CFG.token_budget:,} encoder tokens, "
      f"micro-batch {CFG.micro_batch}")
print(f"Milestones (encoder tokens): {[f'{t:,}' for t in CFG.milestones]}")
if PRESET == "CPU":
    print("⚠️  No GPU detected. This preset only sanity-checks the code. "
          "In Colab: Runtime → Change runtime type → T4 GPU.")

## Step 1 · Data: FineWeb passages split into prefix / middle / suffix

> 🎯 **Paper (§3.1, Fig 2):** *"we take a passage of web text and divide it into three consecutive segments: a prefix, middle, and suffix"* with $n_\text{prefix}=n_\text{middle}=n_\text{suffix}=16$ tokens. The subject model processes **prefix + middle**; activations are read **at the middle tokens** from layer $\ell_\text{read}$. The decoder must then predict the **suffix** from the encoded middle activations. The prefix is never in the loss — it exists only to give the middle tokens context. Because the subject model is instruction-tuned, every passage is prepended with a chat-template prefix (`INSTRUCT_PREFIX`, Appendix A.1).
>
> ✅ **What "right" looks like:** below you'll see a real FineWeb passage split into the three 16-token segments, prepended by the model's chat prefix — exactly the layout of Figure 2.

**Deviations from the paper:**
- 📄 Paper uses the FineWeb corpus (Penedo et al., 2024) / 🔧 We stream the `sample-10BT` subset — same distribution, no 40TB download. 💡 Streaming lets a Colab session start training in seconds.
- 📄 Paper's A.1 gives a hardcoded Llama-3.1 chat prefix / 🔧 We render the *same thing* via `tokenizer.apply_chat_template`, so it stays correct if you switch models (Qwen, 3B, 8B). 💡 The Llama tokenizer inserts the same system-block-with-dates the paper shows.
- 📄 Paper doesn't say *where* in each document the 48-token window is taken / 🔧 We take a random window per document. 💡 Random windows avoid over-sampling document openings; any reasonable choice tests the same claim.

In [ ]:
# ── Step 1.1 · Tokenizer, chat prefix, and window pipeline (§3.1, A.1) ────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, token=HF_TOKEN or None)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.chat_template is None:  # safety net for base models without a chat template
    tokenizer.chat_template = (
        "{% for m in messages %}<|user|>\n{{ m['content'] }}\n{% endfor %}"
    )

# A.1 prepends an INSTRUCT_PREFIX (chat-template system+user header) to every
# passage. We reproduce it for *any* model with a placeholder-split trick:
# render a chat containing a placeholder, keep everything left of it.
_PLACEHOLDER = "<<<PASSAGE>>>"
_rendered = tokenizer.apply_chat_template(
    [{"role": "user", "content": _PLACEHOLDER}], tokenize=False)
INSTRUCT_PREFIX_STR = _rendered.split(_PLACEHOLDER)[0]
INSTRUCT_PREFIX_IDS = torch.tensor(
    tokenizer(INSTRUCT_PREFIX_STR, add_special_tokens=False)["input_ids"])
# Token whose embedding gets *replaced* by soft tokens — its identity is irrelevant (Fig 2 "<dummy str>").
DUMMY_ID = tokenizer.pad_token_id

WINDOW = CFG.n_prefix + CFG.n_middle + CFG.n_suffix  # 48 tokens per example
TRAIN_SKIP_DOCS = 6_000  # first docs of the shuffled stream are reserved for eval/exemplars

def fineweb_windows(skip_docs: int, seed: int):
    """Yield random WINDOW-token windows, one per FineWeb document (§3.1)."""
    from datasets import load_dataset
    ds = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT",
                      split="train", streaming=True)
    ds = ds.shuffle(seed=CFG.seed, buffer_size=10_000)  # fixed corpus order across runs
    rng = random.Random(seed)
    for i, doc in enumerate(ds):
        if i < skip_docs:
            continue
        ids = tokenizer(doc["text"], add_special_tokens=False)["input_ids"]
        if len(ids) < WINDOW:
            continue
        s = rng.randrange(0, len(ids) - WINDOW + 1)
        yield ids[s:s + WINDOW]

def windows_to_batches(window_iter, batch_size: int):
    """Group token windows into (B, WINDOW) int64 tensors."""
    buf = []
    for w in window_iter:
        buf.append(w)
        if len(buf) == batch_size:
            yield torch.tensor(buf, dtype=torch.long)
            buf = []

print(f"INSTRUCT_PREFIX ({len(INSTRUCT_PREFIX_IDS)} tokens):")
print(repr(INSTRUCT_PREFIX_STR))

In [ ]:
# ── Step 1.2 · Cache held-out data (a few minutes, network-bound; cached on disk) ──
# Two held-out sets, disjoint from training (training skips the first 6k docs):
#   eval_windows     — for held-out suffix-loss curves (our version of Fig 3 left)
#   exemplar_windows — for auto-interp exemplars (§3.3)
eval_path, exemplar_path = DATA_DIR / "eval_windows.pt", DATA_DIR / "exemplar_windows.pt"

if eval_path.exists() and exemplar_path.exists():
    EVAL_WINDOWS, EXEMPLAR_WINDOWS = torch.load(eval_path), torch.load(exemplar_path)
    print("Loaded cached held-out windows from disk.")
else:
    gen = fineweb_windows(skip_docs=0, seed=CFG.seed)
    EVAL_WINDOWS = torch.tensor([next(gen) for _ in range(CFG.eval_examples)])
    EXEMPLAR_WINDOWS = torch.tensor([next(gen) for _ in range(CFG.exemplar_docs)])
    torch.save(EVAL_WINDOWS, eval_path); torch.save(EXEMPLAR_WINDOWS, exemplar_path)
print(f"eval: {tuple(EVAL_WINDOWS.shape)}   exemplars: {tuple(EXEMPLAR_WINDOWS.shape)}")

# Show one example the way Figure 2 draws it:
w = EVAL_WINDOWS[0]
print("\n─ example (Fig 2 layout) ─")
print("PREFIX :", repr(tokenizer.decode(w[:CFG.n_prefix])))
print("MIDDLE :", repr(tokenizer.decode(w[CFG.n_prefix:CFG.n_prefix + CFG.n_middle])),
      "← subject activations read here, then encoded to concepts")
print("SUFFIX :", repr(tokenizer.decode(w[-CFG.n_suffix:])),
      "← decoder must predict these tokens")

## Step 2 · The PCD architecture (§2, Fig 2, Eq 1)

> 🎯 **Paper's spec (§2, Eq 1):** at each token position $i$, the encoder computes
>
> $$\mathbf{a}'^{(i)} = \mathbf{W}_\text{emb}\big(\text{TopK}\big(\mathbf{W}_\text{enc}\,\mathbf{a}^{(i)} + \mathbf{b}_\text{enc}\big)\big)$$
>
> with $\mathbf{W}_\text{enc} \in \mathbb{R}^{m\times d}$, $\mathbf{b}_\text{enc} \in \mathbb{R}^m$, $\mathbf{W}_\text{emb} \in \mathbb{R}^{d\times m}$, where $\text{TopK}(\cdot)$ zeroes all but the $k$ largest entries. The decoder $\mathcal{D}$ is *"an LM with the same architecture as $\mathcal{S}$ … identical weights to $\mathcal{S}$ along with a rank-$r$ LoRA adapter"*; the re-embedded activations $\mathbf{a}'$ are *"patched into $\mathcal{D}$'s residual stream at layer $\ell_\text{write}$ as soft tokens, following LatentQA"*, with $\ell_\text{write}=0$. Activations are read at $\ell_\text{read}=15$ of Llama-3.1-8B's 32 layers — the middle of the stack. Initialization (§3.1): decoder = copy of $\mathcal{S}$ + LoRA; $\mathbf{W}_\text{enc}$ random with unit-norm rows; $\mathbf{W}_\text{emb} = \mathbf{W}_\text{enc}^\top$; $\mathbf{b}_\text{enc} = 0$.
>
> ✅ **What "right" looks like:** the smoke-test cell below pushes one batch through subject → encoder → decoder and should print exactly $k=16$ non-zero concepts per token, and a finite suffix loss.

**Why this design matters (worth internalizing before the code):**
- The **encoder never sees the question** and the **decoder never sees raw activations** — the sparse concept list is the *only* channel. That's what makes decoder answers auditable: any prediction traces back to $k$ inspectable concepts (§1).
- $\ell_\text{write}=0$ means the soft tokens *replace embedding-layer outputs* — so we can implement patching with `inputs_embeds`, no hooks needed.

**Deviations from the paper:**
- 📄 Subject = Llama-3.1-8B ($d=4096$, 32 layers), $m=32{,}768$ / 🔧 Llama-3.2-1B ($d=2048$, 16 layers), $m=8d=16{,}384$, $\ell_\text{read}=8$. 💡 Same 8× expansion ratio and same mid-stack read layer, sized for a T4.
- 📄 Two models in memory (subject $\mathcal{S}$ + decoder $\mathcal{D}$) / 🔧 **One** model: since $\mathcal{D} \equiv \mathcal{S}$ + LoRA, we run the subject pass with the LoRA adapter *disabled* and the decoder pass with it *enabled*. 💡 Mathematically identical, halves GPU memory.
- 📄 LoRA rank for the main runs is not stated (Fig 15 sweeps $r\in\{4,8,12,16\}$, "no clear trend"); target modules not stated / 🔧 $r=8$ on all attention+MLP projections.

In [ ]:
# ── Step 2.1 · Load subject model + attach decoder LoRA (~1-3 min) ────────────
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

DTYPE = (torch.bfloat16 if (dev == "cuda" and torch.cuda.is_bf16_supported())
         else torch.float16 if dev == "cuda" else torch.float32)

base = AutoModelForCausalLM.from_pretrained(
    CFG.model_name, torch_dtype=DTYPE, token=HF_TOKEN or None)
base.to(dev)

# §2: decoder = subject weights + rank-r LoRA. A.1: alpha=32, dropout=0.05.
lora_cfg = LoraConfig(
    r=CFG.lora_r, lora_alpha=CFG.lora_alpha, lora_dropout=CFG.lora_dropout,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

# Derived dimensions (§3.1 ratios applied to our model):
CFG.d = model.config.hidden_size
CFG.n_layers = model.config.num_hidden_layers
CFG.m = CFG.expansion * CFG.d                    # 📄 32768 → 🔧 8×d of our model
CFG.l_read = CFG.n_layers // 2                   # 📄 layer 15/32 → 🔧 middle layer
CFG.l_write = 0                                  # unchanged (§3.1)
CFG.k_aux = max(16, round(CFG.m * 500 / 32768))  # 📄 500/32768 ≈ 1.5% of m → 🔧 same ratio
CFG.dead_window = max(50_000, CFG.token_budget // 16)  # 📄 1M of 72M ≈ 1.4% → 🔧 similar ratio, floor 50k
print(f"d={CFG.d}  layers={CFG.n_layers}  m={CFG.m:,}  k={CFG.k}  "
      f"l_read={CFG.l_read}  k_aux={CFG.k_aux}  dead_window={CFG.dead_window:,} tokens")

In [ ]:
# ── Step 2.2 · Encoder (Eq 1), activation reading, soft-token patching (Fig 2) ──

class ConceptEncoder(nn.Module):
    """Eq 1: a' = W_emb( TopK( W_enc a + b_enc ) ).  Kept in fp32 for stability."""

    def __init__(self, d: int, m: int, k: int):
        super().__init__()
        self.k = k
        W = torch.randn(m, d)
        W = W / W.norm(dim=1, keepdim=True)          # §3.1: random rows, unit norm
        self.W_enc = nn.Parameter(W)                 # (m, d)
        self.b_enc = nn.Parameter(torch.zeros(m))    # (m,)  §3.1: init 0
        self.W_emb = nn.Parameter(W.t().clone())     # (d, m) §3.1: init W_enc^T

    def concept_preacts(self, a):
        """Dense concept scores W_enc·a + b_enc, shape (..., m)."""
        return a @ self.W_enc.T + self.b_enc

    def forward(self, a):
        pre = self.concept_preacts(a)                          # (..., m)
        vals, idx = pre.topk(self.k, dim=-1)                   # k largest entries
        sparse = torch.zeros_like(pre).scatter(-1, idx, vals)  # TopK: zero the rest
        a_prime = sparse @ self.W_emb.T                        # (..., d) soft tokens
        return a_prime, idx, vals, pre


@torch.no_grad()
def subject_middle_acts(model, batch):
    """Subject pass (Fig 2 left): feed INSTRUCT_PREFIX + prefix + middle,
    return residual-stream activations at the middle tokens from layer l_read.
    LoRA is disabled -> this is exactly the frozen subject model S."""
    B = batch.shape[0]
    ctx = batch[:, : CFG.n_prefix + CFG.n_middle].to(dev)
    inp = torch.cat([INSTRUCT_PREFIX_IDS.expand(B, -1).to(dev), ctx], dim=1)
    with model.disable_adapter():
        out = model(input_ids=inp, output_hidden_states=True)
    # hidden_states[L] = residual stream after layer L (index 0 = embeddings)
    return out.hidden_states[CFG.l_read][:, -CFG.n_middle:, :].float()


def decoder_suffix_loss(model, a_prime, batch, zero_ablate=False):
    """Decoder pass (Fig 2 right): input = [INSTRUCT_PREFIX][dummy x n_middle][suffix],
    with the dummy embeddings REPLACED by the soft tokens a' (l_write = 0, so
    patching the residual stream at layer 0 == swapping the input embeddings).
    Loss (Eq 2) is next-token prediction on suffix positions only."""
    B = batch.shape[0]
    suffix = batch[:, -CFG.n_suffix:].to(dev)
    prefix = INSTRUCT_PREFIX_IDS.expand(B, -1).to(dev)
    dummy = torch.full((B, CFG.n_middle), DUMMY_ID, device=dev, dtype=torch.long)
    ids = torch.cat([prefix, dummy, suffix], dim=1)

    embeds = model.get_input_embeddings()(ids)
    soft = torch.zeros_like(a_prime) if zero_ablate else a_prime
    p = prefix.shape[1]
    embeds = torch.cat(  # splice the soft tokens over the dummy slots
        [embeds[:, :p], soft.to(embeds.dtype), embeds[:, p + CFG.n_middle:]], dim=1)

    labels = torch.full_like(ids, -100)          # -100 = ignored by the loss
    labels[:, -CFG.n_suffix:] = suffix           # Eq 2: only suffix tokens count
    return model(inputs_embeds=embeds, labels=labels).loss


def amp_ctx():
    return (torch.autocast("cuda", dtype=DTYPE) if dev == "cuda"
            else contextlib.nullcontext())

print("Architecture code ready.")

In [ ]:
# ── Step 2.3 · Smoke test: one batch end-to-end ───────────────────────────────
# Expected: exactly k=16 nonzero concepts/token; a finite loss; and at init the
# soft-token loss ≈ the zero-ablated loss (the decoder hasn't learned to read
# the bottleneck yet — training will open the gap; that gap IS claim C1).
_gen = windows_to_batches(fineweb_windows(TRAIN_SKIP_DOCS, seed=123), 4)
_batch = next(_gen)
_enc = ConceptEncoder(CFG.d, CFG.m, CFG.k).to(dev)

_a = subject_middle_acts(model, _batch)                    # (B, n_middle, d)
_a_prime, _idx, _vals, _pre = _enc(_a)
with amp_ctx():
    _loss = decoder_suffix_loss(model, _a_prime, _batch)
    _loss0 = decoder_suffix_loss(model, _a_prime, _batch, zero_ablate=True)

print(f"subject acts     : {tuple(_a.shape)}  (B, n_middle, d)")
print(f"concept preacts  : {tuple(_pre.shape)}  (B, n_middle, m={CFG.m:,})")
_nonzero = (torch.zeros_like(_pre).scatter(-1, _idx, _vals) != 0).sum(-1)
print(f"active concepts  : {_nonzero.float().mean():.0f} per token "
      f"(top-{CFG.k} of {CFG.m:,} → {100 * CFG.k / CFG.m:.2f}% density)")
print(f"soft tokens      : {tuple(_a_prime.shape)}  → patched at l_write=0")
print(f"suffix loss (soft tokens)   : {_loss.item():.3f}")
print(f"suffix loss (zero-ablated)  : {_loss0.item():.3f}   ← no-information reference")
del _enc, _a, _a_prime, _pre, _gen; gc.collect()
if dev == "cuda":
    torch.cuda.empty_cache()

## Step 3 · The objectives: next-token loss (Eq 2) + auxiliary loss (Eq 3)

> 🎯 **Paper (§3.1, Eq 2):** with segment activations $\mathbf{a}^{(1:n_\text{middle})}$ and suffix tokens $s^{(1:n_\text{suffix})}$:
>
> $$\mathcal{L}_\text{next-token} = -\sum_{t=1}^{n_\text{suffix}} \log p_\mathcal{D}\big(s^{(t)} \mid s^{(1:t-1)},\, \mathcal{E}(\mathbf{a}^{(1:n_\text{middle})})\big)$$
>
> *"At each token position $t$, we can think of $s^{(1:t-1)}$ as the 'question', and $s^{(t)}$ as the corresponding 'answer'."* — next-token prediction as free, unlimited supervision for interpretability.
>
> 🎯 **Paper (§3.2, Eq 3):** many concepts go **dead** (never in the top-$k$) — *"in our training run with 72M tokens, nearly a third of concepts died without intervention"*, and dead concepts score poorly on interpretability (Fig 13 right). Fix: track concepts inactive over the last 1M tokens; for each activation $\mathbf{a}$, take the $k_\text{aux}=500$ inactive concepts with the largest $\mathbf{W}_{\text{enc},i}\cdot\mathbf{a}$ and nudge them toward $\mathbf{a}$:
>
> $$\mathcal{L}_\text{aux} = -\frac{\epsilon_\text{aux}}{k_\text{aux}} \sum_{i \in I} \mathbf{W}_{\text{enc},i} \cdot \mathbf{a}, \qquad \epsilon_\text{aux}=10^{-4}$$
>
> *"This selects dead concepts that are close to being active and nudges them in a direction that encourages them to become active in similar contexts."* With it, **>90% of concepts stay active** at 72M tokens (§3.2, Fig 13 left).
>
> ✅ **Expected once we train (claim C2):** the `use_aux=False` run's active-concept fraction sags well below the `use_aux=True` run's. This is the paper's headline ablation (Fig 3 middle/right + Fig 13), and it's the crispest signal at our small scale.

**Deviations:** 📄 window = 1M tokens, $k_\text{aux}=500$ (of 32,768) / 🔧 window and $k_\text{aux}$ scaled to the same *fractions* of our budget and $m$ (see config printout above). 💡 Dead-concept dynamics depend on relative, not absolute, scale; A.1 notes training *"is not very sensitive"* to these parameters.

In [ ]:
# ── Step 3.1 · Concept-activity tracking + auxiliary loss (§3.2, Eq 3) ────────

class ActivityTracker:
    """A concept is 'dead' if it hasn't appeared in any token's top-k within
    the last `window` encoder tokens (§3.2)."""

    def __init__(self, m, window, device):
        self.window = window
        self.last_active = torch.zeros(m, dtype=torch.long, device=device)
        self.tokens_seen = 0

    def update(self, topk_idx, n_tokens):
        self.last_active[topk_idx.reshape(-1)] = self.tokens_seen + n_tokens
        self.tokens_seen += n_tokens

    def dead_mask(self):
        if self.tokens_seen <= self.window:        # no history yet → nothing is dead
            return torch.zeros_like(self.last_active, dtype=torch.bool)
        return (self.tokens_seen - self.last_active) > self.window

    def active_frac(self):
        return 1.0 - self.dead_mask().float().mean().item()


def aux_loss_fn(encoder, a, pre, dead_mask):
    """Eq 3. `pre` is W_enc·a + b_enc from the encoder forward; Eq 3 uses the
    raw dot product W_enc,i·a, so we subtract the bias. Maximizing that dot
    product (note the minus sign) pulls each selected dead direction toward
    the current activations — reviving it 'in similar contexts'."""
    n_dead = int(dead_mask.sum())
    if n_dead == 0:
        return a.new_zeros(())
    k_aux = min(CFG.k_aux, n_dead)
    pre_nobias = pre - encoder.b_enc                       # (..., m) = W_enc·a
    masked = pre_nobias.masked_fill(~dead_mask, float("-inf"))
    vals, _ = masked.topk(k_aux, dim=-1)                   # top dead dot-products, per token
    return -(CFG.eps_aux / k_aux) * vals.sum(-1).mean()

print("Objectives ready.")

## Step 4 · Pretraining (§3.1, Appendix A.1) — the two runs of the core ablation

> 🎯 **Paper's claim (C1, Fig 3 left):** *"the decoder's loss decreases steadily throughout training, indicating that the encoder learns to pass increasingly useful information through the bottleneck."*
>
> ✅ **Expected below:** held-out suffix loss falls steadily with training tokens, and — the sharper test — falls *increasingly below the zero-ablated reference* (same decoder, soft tokens zeroed). That widening gap is bits of subject-model state flowing through the 16-concept bottleneck.

**Hyperparameters, paper (A.1) vs us:**

| | 📄 Paper | 🔧 Us | 💡 Why |
|---|---|---|---|
| LR / schedule | $10^{-4}$, cosine, no warmup | same | — |
| Weight decay / LoRA α / dropout | 0.01 / 32 / 0.05 | same | — |
| Effective batch | 128 seqs (2048 enc. tokens/step) | 16–32 seqs | fits T4; more optimizer steps per token at same LR |
| Token budgets | separate runs at {9, 18, 36, 72}M | **one run per variant**, metrics + encoder checkpoints at {⅛, ¼, ½, 1}× budget | 8 runs → 2 runs; a mid-cosine checkpoint isn't identical to a fully-annealed short run, but the *trend across milestones* is what we're testing |
| Precision | (not stated; H100-class) | fp16/bf16 autocast, encoder in fp32 | T4 has no usable bf16 |

**Token accounting:** we count *encoder tokens* (= middle tokens, `n_middle` per example), matching how the paper counts SAE-comparable budgets in §3.3.

The next cell defines `train_pcd(use_aux=...)`; the two cells after it launch the **aux** and **no-aux** runs (≈30–50 min each on a T4 — progress bar shows live loss and active-concept fraction; watch the no-aux one decay). Both runs see identical data order, so the ablation is clean. Runs checkpoint at every milestone and skip themselves if already finished.

In [ ]:
# ── Step 4.1 · Training loop (§3.1, A.1) ──────────────────────────────────────
from tqdm.auto import tqdm

def lora_parameters(model):
    return [p for n, p in model.named_parameters() if "lora_" in n]

def reset_lora(model):
    """Re-init LoRA to PEFT defaults so both ablation runs start identically."""
    for n, p in model.named_parameters():
        if "lora_A" in n:
            nn.init.kaiming_uniform_(p, a=math.sqrt(5))
        elif "lora_B" in n:
            nn.init.zeros_(p)

@torch.no_grad()
def eval_suffix_loss(model, encoder):
    """Held-out Eq-2 loss, plus the zero-ablated reference (decoder gets no
    activation information — only INSTRUCT_PREFIX + suffix-so-far)."""
    was_training = model.training
    model.eval()
    tot = tot0 = n = 0
    for i in range(0, len(EVAL_WINDOWS), CFG.micro_batch):
        b = EVAL_WINDOWS[i:i + CFG.micro_batch]
        a = subject_middle_acts(model, b)
        a_prime, *_ = encoder(a)
        with amp_ctx():
            tot += decoder_suffix_loss(model, a_prime, b).item() * len(b)
            tot0 += decoder_suffix_loss(model, a_prime, b, zero_ablate=True).item() * len(b)
        n += len(b)
    if was_training:
        model.train()
    return tot / n, tot0 / n

def record_milestone(run_dir, run_name, encoder, tracker, metrics, log,
                     tokens_seen, next_ms, mpath):
    """Evaluate + checkpoint the encoder at one milestone; return next_ms + 1."""
    el, el0 = eval_suffix_loss(model, encoder)
    metrics["tokens"].append(tokens_seen)
    metrics["eval_loss"].append(el)
    metrics["eval_loss_zero"].append(el0)
    metrics["active_frac"].append(tracker.active_frac())
    sd = {k2: v.detach().half().cpu() for k2, v in encoder.state_dict().items()}
    torch.save(sd, run_dir / f"enc_{CFG.milestones[next_ms]}.pt")
    mpath.write_text(json.dumps(metrics, indent=1))
    (run_dir / "train_log.json").write_text(json.dumps(log))
    print(f"\n[{run_name}] {tokens_seen:,} tokens: eval {el:.3f} "
          f"(zero-ablated {el0:.3f}), active {metrics['active_frac'][-1]:.2f}")
    return next_ms + 1


def train_pcd(use_aux: bool, run_name: str):
    run_dir = RUN_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    mpath = run_dir / "metrics.json"
    if mpath.exists() and len(json.loads(mpath.read_text())["tokens"]) >= len(CFG.milestones):
        print(f"[{run_name}] already trained — delete {run_dir} to retrain.")
        return

    set_seed(CFG.seed)                       # same init + data order for both variants
    encoder = ConceptEncoder(CFG.d, CFG.m, CFG.k).to(dev)   # fp32
    reset_lora(model)
    tracker = ActivityTracker(CFG.m, CFG.dead_window, dev)

    tokens_per_step = CFG.micro_batch * CFG.n_middle        # encoder tokens per step
    total_steps = math.ceil(CFG.token_budget / tokens_per_step)  # ceil: final milestone must fire
    opt = torch.optim.AdamW(
        [{"params": encoder.parameters()}, {"params": lora_parameters(model)}],
        lr=CFG.lr, weight_decay=CFG.weight_decay)           # A.1
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps)  # A.1: cosine, no warmup
    scaler = torch.amp.GradScaler(enabled=(dev == "cuda" and DTYPE == torch.float16))

    stream = windows_to_batches(fineweb_windows(TRAIN_SKIP_DOCS, seed=CFG.seed), CFG.micro_batch)
    metrics = {"run": run_name, "use_aux": use_aux, "tokens": [],
               "eval_loss": [], "eval_loss_zero": [], "active_frac": []}
    log, tokens_seen, next_ms = [], 0, 0

    model.train()
    pbar = tqdm(range(total_steps), desc=run_name)
    for step in pbar:
        batch = next(stream)
        a = subject_middle_acts(model, batch)               # frozen S, no grad
        a_prime, idx, vals, pre = encoder(a)                # Eq 1 (fp32)
        dead = tracker.dead_mask()                          # §3.2 (before this batch)
        tracker.update(idx, a.shape[0] * a.shape[1])

        with amp_ctx():
            loss_nt = decoder_suffix_loss(model, a_prime, batch)   # Eq 2
        loss = loss_nt + (aux_loss_fn(encoder, a, pre, dead) if use_aux else 0.0)  # Eq 3

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        tokens_seen += a.shape[0] * a.shape[1]

        if step % 50 == 0:
            af = tracker.active_frac()
            pbar.set_postfix(loss=f"{loss_nt.item():.3f}", active=f"{af:.2f}")
            log.append({"tokens": tokens_seen, "loss": loss_nt.item(), "active_frac": af})

        if next_ms < len(CFG.milestones) and tokens_seen >= CFG.milestones[next_ms]:
            next_ms = record_milestone(run_dir, run_name, encoder, tracker, metrics,
                                       log, tokens_seen, next_ms, mpath)

    while next_ms < len(CFG.milestones):  # safety: flush any un-hit milestone
        next_ms = record_milestone(run_dir, run_name, encoder, tracker, metrics,
                                   log, tokens_seen, next_ms, mpath)

    model.save_pretrained(run_dir / "lora_final")           # decoder adapter, for Step 8
    del encoder
    gc.collect()
    if dev == "cuda":
        torch.cuda.empty_cache()
    print(f"[{run_name}] done: {tokens_seen:,} encoder tokens, {total_steps} steps.")

print("Trainer ready.")

In [ ]:
# ── Step 4.2 · RUN 1: PCD *with* auxiliary loss (~30-50 min on T4) ────────────
train_pcd(use_aux=True, run_name="aux")

In [ ]:
# ── Step 4.3 · RUN 2: ablation *without* auxiliary loss (~30-50 min on T4) ────
# Same seed, same data order — the ONLY difference is Eq 3. (Fig 3 / Fig 13 ablation.)
train_pcd(use_aux=False, run_name="noaux")

## Step 5 · Result 1: predictive scaling (C1) + concept survival (C2)

> 🎯 **Paper's expected picture:**
> - **Fig 3 (left):** training loss on FineWeb decreases steadily with tokens for every budget — the encoder passes increasingly useful information through the bottleneck.
> - **Fig 13 (left):** without the aux loss the number of active concepts collapses (→ ~⅔ of 32,768 by 72M tokens); with it, >90% stay active.
>
> ✅ **Our plots should show:** (1) train + held-out suffix loss falling with tokens; (2) held-out loss *below* the zero-ablated reference, with the gap growing — information genuinely flows through the 16-concept channel; (3) active-concept fraction: aux ≫ no-aux by the end of training.
>
> ⚠️ If the aux/no-aux *suffix-loss* curves look nearly identical — that's consistent with the paper: the aux loss barely changes predictive loss (its coefficient is tiny); it exists to protect *interpretability* metrics (Fig 3 middle/right), which we measure in Steps 6–7.

In [ ]:
# ── Step 5.1 · Plot our Fig-3(left) + Fig-13(left) analogues ─────────────────
import matplotlib.pyplot as plt

M = {name: json.loads((RUN_DIR / name / "metrics.json").read_text())
     for name in ["aux", "noaux"]}
L = {name: json.loads((RUN_DIR / name / "train_log.json").read_text())
     for name in ["aux", "noaux"]}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]  # ~ paper Fig 3, first column
for n in M:
    t = [r["tokens"] for r in L[n]][1:]
    y = [r["loss"] for r in L[n]][1:]
    k = max(1, len(y) // 40)  # light smoothing
    ys = np.convolve(y, np.ones(k) / k, mode="valid")
    ax.plot(t[k - 1:], ys, color=COL[n], label=LBL[n], lw=1.5)
ax.set(xscale="log", xlabel="encoder tokens", ylabel="train suffix loss",
       title="Training loss on FineWeb\n(cf. paper Fig 3, col 1)")
ax.legend()

ax = axes[1]
for n in M:
    ax.plot(M[n]["tokens"], M[n]["eval_loss"], "o-", color=COL[n], label=LBL[n])
ax.plot(M["aux"]["tokens"], M["aux"]["eval_loss_zero"], "s--", color="gray",
        label="zero-ablated soft tokens\n(no activation info)")
ax.set(xscale="log", xlabel="encoder tokens", ylabel="held-out suffix loss",
       title="Held-out loss vs no-information reference\n(gap = bits through the bottleneck)")
ax.legend()

ax = axes[2]  # ~ paper Fig 13, left
for n in M:
    t = [r["tokens"] for r in L[n]]
    ax.plot(t, [r["active_frac"] for r in L[n]], color=COL[n], label=LBL[n])
ax.axhline(0.9, color="gray", ls=":", lw=1)
ax.set(xlabel="encoder tokens", ylabel="fraction of concepts active",
       ylim=(0, 1.05), title="Concept survival\n(cf. paper Fig 13, left)")
ax.legend()

plt.tight_layout()
plt.savefig(RUN_DIR / "result1_scaling.png", dpi=120)
plt.show()

gap = [z - e for z, e in zip(M["aux"]["eval_loss_zero"], M["aux"]["eval_loss"])]
print(f"C1 — bottleneck information gap by milestone: "
      f"{[f'{g:+.3f}' for g in gap]}  (should grow ↑)")
print(f"C2 — final active fraction: aux={M['aux']['active_frac'][-1]:.2f}  "
      f"vs no-aux={M['noaux']['active_frac'][-1]:.2f}  (paper: >0.90 vs ~0.67)")

## Step 6 · Concept precision: auto-interp score (§3.3, Fig 3 middle)

> 🎯 **Paper (§3.3):** *"For each concept direction, we collect top-activating exemplars from held-out FineWeb passages, generate natural language descriptions, and measure how well a finetuned simulator can predict activation patterns on new exemplars given only the description (via Pearson correlation). We evaluate a random sample of 400 concepts."* (Pipeline of Choi et al., 2024.)
>
> ✅ **Expected result (C3-precision, Fig 3 middle):** auto-interp score *increases with training tokens* for the aux run; the no-aux run **plateaus or declines** on longer training. Mechanism (Fig 13 right): dead concepts are disproportionately uninterpretable, and the no-aux run accumulates dead concepts.
>
> ⚠️ **Expectation management:** the paper's aux/no-aux divergence emerges between 36M→72M tokens; our budget ends near their *first* x-tick. Expect the aux run to trend up; the gap to no-aux may be modest — the cleaner small-scale evidence for C3 is the combination *(dead fraction ↑ in Step 5) × (dead concepts score poorly, visible in the scatter below)*.

**Deviations:**
- 📄 400 concepts, description + *finetuned* simulator model / 🔧 64 random concepts, **Claude Haiku** as both describer and (zero-shot) simulator, rating whole snippets 0–10 instead of per-token. 💡 One API call per concept per role ≈ **$1–3 total**; zero-shot simulation is noisier but unbiased between the two runs, which is all a *comparison* needs.
- 🔧 **No-API fallback:** a crude "top-token consistency" heuristic (do the top exemplars share the same max-activating token string?). It measures monosemanticity-of-token, not meaning — treat it as a sanity signal only.

Set your key in the next cell (Colab: 🔑 Secrets sidebar → `ANTHROPIC_API_KEY`). Without a key the notebook still runs end-to-end on the heuristic.

In [ ]:
# ── Step 6.1 · 🔑 ANTHROPIC_API_KEY (optional but recommended) ────────────────
# In Colab: click the key icon (left sidebar) → add secret ANTHROPIC_API_KEY → toggle
# notebook access. Or paste it below for a quick session (never commit a key!).
ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY")
# ANTHROPIC_API_KEY = "sk-ant-..."   # ← manual override

USE_CLAUDE = bool(ANTHROPIC_API_KEY)
CLAUDE_MODEL = "claude-haiku-4-5-20251001"   # cheap + fine as a judge; swap for a bigger model if you like
print("Claude auto-interp scoring:", "ENABLED ✅" if USE_CLAUDE
      else "disabled → using heuristic fallback only")

In [ ]:
# ── Step 6.2 · Exemplar collection over held-out FineWeb (§3.3) ───────────────
# The subject model is frozen, so its activations on the exemplar corpus are
# computed ONCE and reused for every checkpoint of both runs.

@torch.no_grad()
def subject_window_acts(windows, batch_size=32):
    """(N, T) token windows → (N, T, d) layer-l_read activations (fp16, CPU)."""
    model.eval()
    outs = []
    for i in tqdm(range(0, len(windows), batch_size), desc="subject acts", leave=False):
        b = windows[i:i + batch_size].to(dev)
        inp = torch.cat([INSTRUCT_PREFIX_IDS.expand(len(b), -1).to(dev), b], dim=1)
        with model.disable_adapter(), amp_ctx():
            hs = model(input_ids=inp, output_hidden_states=True).hidden_states[CFG.l_read]
        outs.append(hs[:, -windows.shape[1]:, :].half().cpu())
    return torch.cat(outs)

acts_path = DATA_DIR / "exemplar_acts.pt"
if acts_path.exists():
    EXEMPLAR_ACTS = torch.load(acts_path)
else:
    EXEMPLAR_ACTS = subject_window_acts(EXEMPLAR_WINDOWS)   # ~1-3 min
    torch.save(EXEMPLAR_ACTS, acts_path)
print("exemplar activations:", tuple(EXEMPLAR_ACTS.shape))

# 📄 Paper scores a random sample of 400 concepts → 🔧 64 (same ids across checkpoints).
CONCEPT_IDS = np.sort(np.random.default_rng(0).choice(CFG.m, CFG.autointerp_concepts,
                                                      replace=False))

def load_encoder(path):
    sd = torch.load(path, map_location="cpu")
    enc = ConceptEncoder(CFG.d, CFG.m, CFG.k)
    enc.load_state_dict({k: v.float() for k, v in sd.items()})
    return enc.to(dev).eval()

@torch.no_grad()
def concept_acts(enc, concept_ids, acts_cpu=None):
    """Pre-activations W_enc·a + b for selected concepts on the exemplar corpus.
    Returns (N, T, C) fp32 on CPU."""
    acts_cpu = EXEMPLAR_ACTS if acts_cpu is None else acts_cpu
    ids = torch.as_tensor(np.asarray(concept_ids), dtype=torch.long)
    W = enc.W_enc[ids].T.to(dev).float()
    b = enc.b_enc[ids].to(dev).float()
    outs = []
    for i in range(0, len(acts_cpu), 256):
        a = acts_cpu[i:i + 256].to(dev).float()
        outs.append((a @ W + b).cpu())
    return torch.cat(outs)

def build_exemplars(cacts, top_n=10):
    """Per concept: the top_n windows by max token activation (§3.3)."""
    wmax = cacts.amax(dim=1)                       # (N, C) per-window peak act
    top_vals, top_idx = wmax.topk(top_n, dim=0)    # (top_n, C)
    out = []
    for j in range(cacts.shape[-1]):
        rows = top_idx[:, j]
        out.append({"windows": EXEMPLAR_WINDOWS[rows],   # (top_n, T) token ids
                    "acts": cacts[rows, :, j],           # (top_n, T) activations
                    "wmax": wmax[:, j]})                 # (N,) all-window peaks
    return out

def render_snippet(window_ids, tok_acts=None, mark=False):
    """Decode a window token-by-token; wrap strongly-activating tokens in << >>."""
    pieces = [tokenizer.decode([t]) for t in window_ids.tolist()]
    if mark and tok_acts is not None and tok_acts.max() > 0:
        thr = 0.6 * tok_acts.max()
        pieces = [f"<<{p}>>" if v >= thr else p for p, v in zip(pieces, tok_acts.tolist())]
    return "".join(pieces).replace("\n", " ")

print(f"Scoring {len(CONCEPT_IDS)} concepts per checkpoint "
      f"(paper: 400) — ids fixed across checkpoints.")

In [ ]:
# ── Step 6.3 · Scorers: Claude describe→simulate→Pearson, + heuristic fallback ──
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

def heuristic_score(ex):
    """Fallback precision proxy: fraction of top exemplars sharing the same
    max-activating token string. Crude — measures token-level consistency only."""
    toks = [tokenizer.decode([w[int(a.argmax())]]).strip().lower()
            for w, a in zip(ex["windows"][:8], ex["acts"][:8])]
    return Counter(toks).most_common(1)[0][1] / len(toks)

_CLIENT = None
def claude(prompt, max_tokens=300):
    global _CLIENT
    import anthropic
    if _CLIENT is None:
        _CLIENT = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    for attempt in range(4):
        try:
            r = _CLIENT.messages.create(model=CLAUDE_MODEL, max_tokens=max_tokens,
                                        messages=[{"role": "user", "content": prompt}])
            return r.content[0].text
        except Exception:
            time.sleep(2 ** attempt)
    return ""

def claude_score_concept(ex, rng):
    """§3.3 pipeline, miniaturized:
    describe from exemplars 1-5 → simulate on exemplars 6-10 + 5 random windows
    → Pearson(simulated, true peak activation)."""
    shown = "\n".join(f"{i+1}. {render_snippet(w, a, mark=True)}"
                      for i, (w, a) in enumerate(zip(ex["windows"][:5], ex["acts"][:5])))
    desc = claude(
        "You are analyzing one 'concept' direction inside a language model. Below are "
        "the text snippets where it activates most strongly; tokens where it fires are "
        f"marked like <<this>>.\n\n{shown}\n\n"
        "Reply with ONE sentence describing what pattern this concept detects. "
        "No preamble.", max_tokens=100).strip()
    if not desc:
        return 0.0, ""

    n_test = 5
    rand_rows = torch.tensor(rng.choice(len(ex["wmax"]), n_test, replace=False))
    test_windows = torch.cat([ex["windows"][5:10], EXEMPLAR_WINDOWS[rand_rows]])
    truth = torch.cat([ex["wmax"].topk(10).values[5:10], ex["wmax"][rand_rows]]).numpy()
    snips = "\n".join(f"{i+1}. {render_snippet(w)}" for i, w in enumerate(test_windows))
    reply = claude(
        f'A concept direction in a language model is described as: "{desc}"\n\n'
        f"For each numbered snippet, rate 0-10 how strongly this concept would activate "
        f"anywhere in the snippet.\n\n{snips}\n\n"
        f"Reply with ONLY a JSON list of {len(test_windows)} numbers.", max_tokens=120)
    try:
        sim = np.array(json.loads(reply[reply.index("["):reply.index("]") + 1]),
                       dtype=float)
        assert len(sim) == len(truth)
    except Exception:
        return 0.0, desc
    if sim.std() == 0 or truth.std() == 0:
        return 0.0, desc
    return float(np.corrcoef(sim, truth)[0, 1]), desc

def autointerp_checkpoint(run_name, ms_tokens):
    enc = load_encoder(RUN_DIR / run_name / f"enc_{ms_tokens}.pt")
    exs = build_exemplars(concept_acts(enc, CONCEPT_IDS))
    del enc
    res = {"heuristic": float(np.mean([heuristic_score(e) for e in exs]))}
    if USE_CLAUDE:
        rngs = [np.random.default_rng(1000 + i) for i in range(len(exs))]
        with ThreadPoolExecutor(8) as pool:
            scored = list(pool.map(lambda t: claude_score_concept(*t), zip(exs, rngs)))
        res["claude_scores"] = [s for s, _ in scored]
        res["claude"] = float(np.mean(res["claude_scores"]))
        res["descs"] = [d for _, d in scored]
    return res

print("Scorers ready.")

In [ ]:
# ── Step 6.4 · Score every checkpoint × both runs, then plot (Fig 3-middle analogue) ──
# Cost: 4 milestones × 2 runs × 64 concepts × 2 Haiku calls ≈ 1000 calls ≈ $1-2, ~5-10 min.
import matplotlib.pyplot as plt

ai_path = RUN_DIR / "autointerp.json"
AUTOINTERP = json.loads(ai_path.read_text()) if ai_path.exists() else {}

for run_name in ["aux", "noaux"]:
    for t in CFG.milestones:
        key = f"{run_name}/{t}"
        if key not in AUTOINTERP:
            print("scoring", key, "...")
            AUTOINTERP[key] = autointerp_checkpoint(run_name, t)
            ai_path.write_text(json.dumps(AUTOINTERP, indent=1))

fig, axes = plt.subplots(1, 2 if USE_CLAUDE else 1, figsize=(11 if USE_CLAUDE else 6, 4),
                         squeeze=False)
metrics_to_plot = ([("claude", "auto-interp score (Pearson × 100)", 100.0)]
                   if USE_CLAUDE else []) + \
                  [("heuristic", "heuristic token-consistency (0-1)", 1.0)]
for ax, (mk, ylabel, scale) in zip(axes[0], metrics_to_plot):
    for run_name in ["aux", "noaux"]:
        ys = [AUTOINTERP[f"{run_name}/{t}"][mk] * scale for t in CFG.milestones]
        ax.plot(CFG.milestones, ys, "o-", color=COL[run_name], label=LBL[run_name])
    ax.set(xscale="log", xlabel="encoder tokens", ylabel=ylabel,
           title="Concept precision\n(cf. paper Fig 3, col 2)")
    ax.legend()
plt.tight_layout(); plt.savefig(RUN_DIR / "result2_autointerp.png", dpi=120); plt.show()

if USE_CLAUDE:
    # Fig 13 (right) analogue: are DEAD concepts less interpretable? Uses the final
    # aux checkpoint's activity: approximate 'dead' as near-constant exemplar acts.
    final = AUTOINTERP[f"noaux/{CFG.milestones[-1]}"]
    enc = load_encoder(RUN_DIR / "noaux" / f"enc_{CFG.milestones[-1]}.pt")
    spread = concept_acts(enc, CONCEPT_IDS).amax(1).std(0).numpy()  # activity spread per concept
    plt.figure(figsize=(5, 3.5))
    plt.scatter(spread, np.array(final["claude_scores"]) * 100, s=14)
    plt.xscale("log"); plt.xlabel("concept activity spread (proxy for frequency)")
    plt.ylabel("auto-interp score"); plt.title("Low-activity concepts score poorly\n(cf. paper Fig 13, right)")
    plt.tight_layout(); plt.show()
    # Peek at a few descriptions — this is your dictionary! (paper Table 1 vibes)
    best = np.argsort(final["claude_scores"])[::-1][:5]
    print("Top-scoring concepts (no-aux final ckpt):")
    for j in best:
        print(f"  [{CONCEPT_IDS[j]}] {final['claude_scores'][j]:.2f} — {final['descs'][j]}")

## Step 7 · Concept recall: does *some* concept capture each attribute? (§3.3, Fig 3 right)

> 🎯 **Paper (§3.3):** *"To measure concept coverage, we use the SynthSys dataset (Choi et al., 2025), which defines user attributes (e.g. 'marital status') … For each attribute, we construct a balanced binary classification task and train a scalar linear classifier (i.e. a threshold) on each encoder direction's activations. We report the best classifier's held-out accuracy, measuring whether **some** concept captures each attribute."*
>
> ✅ **Expected result (C3-recall, Fig 3 right):** recall improves with pretraining data for the aux run; **without the aux loss, recall decreases as training progresses** (their clearest ablation signal, visible already at 36M→72M).

**Deviations:**
- 📄 SynthSys user attributes (80 attributes; dataset not publicly released) / 🔧 4 topic "attributes" from **AG News** (World / Sports / Business / Sci-Tech), one-vs-rest, balanced 128 pos + 128 neg, half train / half test. 💡 Same *measurement*: is there at least one encoder direction whose thresholded activation classifies the attribute? Only the attribute source differs.
- 🔧 Feature per (text, direction) = **max over tokens** of the concept pre-activation, matching how exemplars are ranked in §3.3.

The threshold-classifier fit is fully vectorized over all $m$ directions and includes a brute-force self-check.

In [ ]:
# ── Step 7.1 · Recall proxy: best single-concept threshold classifier per attribute ──
import matplotlib.pyplot as plt
from datasets import load_dataset

tokenizer.padding_side = "right"  # pads must go AFTER the text (we prepend the chat prefix)

def fit_direction_thresholds(f, y):
    """For each direction (column of f: (N, m)), the best train-accuracy threshold
    classifier  predict = sign * (x - thr) > 0. Vectorized over all m directions."""
    N, m = f.shape
    fs, order = f.sort(dim=0)
    ys = y[order]                                   # labels sorted by feature value
    cpos = ys.cumsum(0)
    P = y.sum()
    ranks = torch.arange(1, N + 1, device=f.device, dtype=f.dtype).unsqueeze(1)
    corr_hi = (P - cpos) + (ranks - cpos)           # cut after rank i, positives above
    corr_hi = torch.cat([torch.full((1, m), float(P), device=f.device), corr_hi])  # cut 0
    best = torch.maximum(corr_hi, N - corr_hi).argmax(0)          # (m,) best cut, either polarity
    lo = torch.cat([fs[:1] - 1.0, fs])              # feature value just below each cut
    hi = torch.cat([fs, fs[-1:] + 1.0])             # feature value just above each cut
    ar = torch.arange(m, device=f.device)
    thr = (lo[best, ar] + hi[best, ar]) / 2
    sign = torch.where(corr_hi[best, ar] >= N - corr_hi[best, ar], 1.0, -1.0)
    return thr, sign

# self-check vs brute force on tiny random data
_f = torch.randn(40, 7); _y = (torch.rand(40) > 0.5).float()
_thr, _sign = fit_direction_thresholds(_f, _y)
_acc = ((_sign * (_f - _thr) > 0).float() == _y.unsqueeze(1)).float().mean(0)
for _j in range(7):
    _brute = max(((_s * (_f[:, _j] - _t) > 0).float() == _y).float().mean().item()
                 for _t in _f[:, _j].tolist() + [_f[:, _j].min() - 1] for _s in (1, -1))
    assert abs(_acc[_j].item() - _brute) < 1e-6, "threshold fit mismatch!"
print("threshold-classifier self-check passed ✓")

# Balanced attribute data (📄 SynthSys → 🔧 AG News topics)
AG_CLASSES = ["World", "Sports", "Business", "Sci/Tech"]
_ag = load_dataset("fancyzhx/ag_news", split="train").shuffle(seed=0)
_by_class = {c: [] for c in range(4)}
for ex in _ag:
    if len(_by_class[ex["label"]]) < 128:
        _by_class[ex["label"]].append(ex["text"])
    if all(len(v) >= 128 for v in _by_class.values()):
        break
AG_TEXTS = sum(_by_class.values(), [])
AG_LABELS = torch.tensor(sum([[c] * 128 for c in range(4)], []))

@torch.no_grad()
def subject_text_feats(enc):
    """(len(texts), m) features: max-over-tokens concept pre-activation per text."""
    W = enc.W_enc.T.to(dev).float()
    feats = []
    model.eval()
    for i in range(0, len(AG_TEXTS), 16):
        tt = tokenizer(AG_TEXTS[i:i + 16], return_tensors="pt", padding=True,
                       truncation=True, max_length=64, add_special_tokens=False)
        B, T = tt.input_ids.shape
        inp = torch.cat([INSTRUCT_PREFIX_IDS.expand(B, -1), tt.input_ids], 1).to(dev)
        mask = torch.cat([torch.ones(B, INSTRUCT_PREFIX_IDS.shape[0], dtype=torch.long),
                          tt.attention_mask], 1).to(dev)
        with model.disable_adapter(), amp_ctx():
            hs = model(input_ids=inp, attention_mask=mask,
                       output_hidden_states=True).hidden_states[CFG.l_read]
        pre = hs[:, -T:, :].float() @ W                       # (B, T, m); bias irrelevant to thresholds
        pre = pre.masked_fill(~tt.attention_mask.bool().unsqueeze(-1).to(dev), float("-inf"))
        feats.append(pre.amax(1).cpu())
    return torch.cat(feats)

def concept_recall(enc, rng):
    feats = subject_text_feats(enc).to(dev)
    accs, winners = [], []
    for c in range(4):
        pos = torch.where(AG_LABELS == c)[0]
        neg = torch.where(AG_LABELS != c)[0]
        neg = neg[torch.as_tensor(rng.choice(len(neg), 128, replace=False))]
        idx = torch.cat([pos, neg])
        y = torch.cat([torch.ones(128), torch.zeros(128)])
        perm = torch.as_tensor(rng.permutation(256))
        idx, y = idx[perm], y[perm].to(dev)
        f = feats[idx]
        tr, te = slice(0, 128), slice(128, 256)
        thr, sign = fit_direction_thresholds(f[tr], y[tr])
        acc = ((sign * (f[te] - thr) > 0).float() == y[te].unsqueeze(1)).float().mean(0)
        accs.append(acc.max().item()); winners.append(int(acc.argmax()))
    return accs, winners

recall_path = RUN_DIR / "recall.json"
RECALL = json.loads(recall_path.read_text()) if recall_path.exists() else {}
for run_name in ["aux", "noaux"]:
    for t in CFG.milestones:
        key = f"{run_name}/{t}"
        if key not in RECALL:
            enc = load_encoder(RUN_DIR / run_name / f"enc_{t}.pt")
            accs, winners = concept_recall(enc, np.random.default_rng(7))
            RECALL[key] = {"accs": accs, "winners": winners}
            del enc
            print(f"{key}: mean recall {np.mean(accs):.3f}")
recall_path.write_text(json.dumps(RECALL, indent=1))

plt.figure(figsize=(6, 4))
for run_name in ["aux", "noaux"]:
    ys = [np.mean(RECALL[f"{run_name}/{t}"]["accs"]) for t in CFG.milestones]
    plt.plot(CFG.milestones, ys, "o-", color=COL[run_name], label=LBL[run_name])
plt.axhline(0.5, color="gray", ls=":", lw=1, label="chance")
plt.xscale("log"); plt.xlabel("encoder tokens"); plt.ylabel("concept recall (best-direction acc)")
plt.title("Concept recall on topic attributes\n(cf. paper Fig 3, col 3)")
plt.legend(); plt.tight_layout()
plt.savefig(RUN_DIR / "result3_recall.png", dpi=120); plt.show()

# Which concept "won" each attribute? Show its top exemplar tokens (final aux ckpt).
enc = load_encoder(RUN_DIR / "aux" / f"enc_{CFG.milestones[-1]}.pt")
final = RECALL[f"aux/{CFG.milestones[-1]}"]
for c, (attr, acc, w) in enumerate(zip(AG_CLASSES, final["accs"], final["winners"])):
    ex = build_exemplars(concept_acts(enc, [w]))[0]
    toks = [tokenizer.decode([wd[int(a.argmax())]]).strip() for wd, a in
            zip(ex["windows"][:6], ex["acts"][:6])]
    print(f"{attr:9s} acc={acc:.2f}  concept #{w}: top tokens {toks}")
del enc

## Step 8 (optional) · Mini case study: detecting an implanted concept (§5.3, Fig 11)

> 🎯 **Paper (§5.3):** derive a steering vector $v$ for a concept from the activation difference between a paragraph about the concept and an unrelated one; steer the subject model's activations toward $v$ (strength 3); then check whether the *decoder verbalizes the injected concept* and whether *encoder concepts related to it light up*. PCDs verbalize injected concepts far more often than prompting the subject model itself (Fig 11).
>
> ✅ **Expected here (qualitative only):** after steering, (1) the encoder's top-$k$ concept list should shift toward concepts whose exemplars are ocean/water related, and (2) the decoder's *continuations* should drift ocean-ward relative to unsteered ones.

**Big honest deviations — read before trusting anything below:**
- 📄 The paper's decoder was **finetuned for question-answering** on SynthSys (§4) before the case studies; we **skipped §4 entirely** (SynthSys isn't public). Our decoder only knows the pretraining task — *continue the text* — so instead of asking it questions we compare its **continuations** with and without steering. Same claim (information about the implanted concept crosses the bottleneck), weaker instrument.
- 📄 Steering "strength 3" with an unspecified normalization / 🔧 we scale the unit steering vector to a multiple of the typical activation norm. Try a couple of `ALPHA` values.
- At a ~1M-token budget this may simply fail — the paper itself shows this capability *emerging with scale* (Fig 11 left). A null result here is not evidence against the paper.

In [ ]:
# ── Step 8.1 · Implant "ocean" into the activations; watch the bottleneck ─────
# Use the aux run's final decoder adapter:
try:
    model.load_adapter(str(RUN_DIR / "aux" / "lora_final"), adapter_name="aux_final")
except Exception:
    pass  # already loaded
model.set_adapter("aux_final")
model.eval()

OCEAN_TEXTS = [
    "The ocean stretched to the horizon, waves crashing against the rocky shore as gulls wheeled overhead.",
    "Deep beneath the sea surface, schools of fish drifted past coral reefs swaying in the current.",
    "The tide rolled in slowly, salt spray misting over the beach while sailboats bobbed in the harbor.",
    "Marine biologists study whales, dolphins, and the vast underwater ecosystems of the deep ocean.",
]
NEUTRAL_TEXTS = [
    "The quarterly report showed steady growth in revenue across all three regional divisions.",
    "She adjusted the recipe, adding flour gradually until the dough reached the right consistency.",
    "The committee reviewed the proposal and scheduled a follow-up meeting for next Tuesday.",
    "Traffic on the highway slowed near the construction zone during the morning commute.",
]

@torch.no_grad()
def mean_act(texts):
    vs = []
    for t in texts:
        ids = torch.tensor([tokenizer(t, add_special_tokens=False)["input_ids"]])
        inp = torch.cat([INSTRUCT_PREFIX_IDS.unsqueeze(0), ids], 1).to(dev)
        with model.disable_adapter(), amp_ctx():
            hs = model(input_ids=inp, output_hidden_states=True).hidden_states[CFG.l_read]
        vs.append(hs[0, -ids.shape[1]:, :].float().mean(0))
    return torch.stack(vs).mean(0)

v = mean_act(OCEAN_TEXTS) - mean_act(NEUTRAL_TEXTS)      # §5.3 contrastive steering vector
v = v / v.norm()

encoder = load_encoder(RUN_DIR / "aux" / f"enc_{CFG.milestones[-1]}.pt")
base_batch = EVAL_WINDOWS[1:2]                            # an arbitrary neutral passage
a = subject_middle_acts(model, base_batch)                # (1, n_middle, d)

def top_concept_report(a_batch, label):
    _, idx, vals, _ = encoder(a_batch)
    ids, counts = np.unique(idx.cpu().numpy(), return_counts=True)
    top_ids = ids[np.argsort(-counts)][:8]
    print(f"\n{label} — most common top-k concepts across the {CFG.n_middle} middle tokens:")
    exs = build_exemplars(concept_acts(encoder, top_ids))
    for cid, ex in zip(top_ids, exs):
        toks = [tokenizer.decode([w[int(t.argmax())]]).strip() for w, t in
                zip(ex["windows"][:5], ex["acts"][:5])]
        print(f"  #{cid:5d}  exemplar tokens: {toks}")

@torch.no_grad()
def decoder_continuations(a_prime, n=4, max_new_tokens=40):
    prefix = INSTRUCT_PREFIX_IDS.expand(n, -1).to(dev)
    dummy = torch.full((n, CFG.n_middle), DUMMY_ID, device=dev, dtype=torch.long)
    ids = torch.cat([prefix, dummy], 1)
    embeds = model.get_input_embeddings()(ids)
    embeds = torch.cat([embeds[:, :prefix.shape[1]],
                        a_prime.expand(n, -1, -1).to(embeds.dtype)], 1)
    out = model.generate(inputs_embeds=embeds,
                         attention_mask=torch.ones_like(ids),
                         do_sample=True, temperature=0.8, max_new_tokens=max_new_tokens,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(out, skip_special_tokens=True)

OCEAN_WORDS = ["ocean", "sea", "wave", "water", "beach", "tide", "marine",
               "ship", "fish", "coral", "shore", "sail"]
def ocean_rate(texts):
    return np.mean([any(w in t.lower() for w in OCEAN_WORDS) for t in texts])

for ALPHA in [0.0, 1.0, 2.0]:
    a_s = a + ALPHA * v * a.norm(dim=-1, keepdim=True)    # 📄 "strength 3", norm unspecified
    a_prime, *_ = encoder(a_s)
    top_concept_report(a_s, f"ALPHA={ALPHA}")
    gens = decoder_continuations(a_prime)
    print(f"  decoder continuations (ocean-word rate {ocean_rate(gens):.0%}):")
    for g in gens:
        print(f"   · {g[:120]}")

## Step 9 · What I should take away

### Claims tested vs skipped

| Paper claim | Tested here? | Verdict source |
|---|---|---|
| **C1** — decoder steadily improves at predicting suffix tokens from bottlenecked activations (§3.1, Fig 3 left) | ✅ tested | Step 5: falling held-out loss **and** a growing gap vs the zero-ablated reference |
| **C2** — concepts die without the aux loss; Eq 3 keeps >90% alive (§3.2, Fig 13) | ✅ tested — the crispest signal at small scale | Step 5, right panel |
| **C3** — precision (auto-interp) & recall improve with data *with* aux, plateau/decline *without* (§3.3, Fig 3 mid/right) | ⚠️ tested in miniature | Steps 6–7; our budget ends near the paper's *first* x-tick, so treat direction, not magnitude |
| Implanted concepts surface through the bottleneck (§5.3, Fig 11) | ⚠️ qualitative mini-probe only | Step 8, with a non-finetuned decoder |
| Decoder QA (SynthSys finetuning, §4) | ❌ skipped — SynthSys not public; QA finetuning is a second training phase | — |
| SAE / KL-SAE baseline comparisons (§3.3, Fig 4) | ❌ skipped — 6 extra training runs | a natural extension: the TopK-SAE trainer is ~30 lines away from our `ConceptEncoder` |
| Jailbreak / secret-hint case studies (§5.1–5.2) | ❌ skipped — needs the QA-finetuned decoder + curated attacks | — |

### What the small-scale result does establish
- The **architecture trains stably end-to-end**: gradients flow from next-token prediction, through soft-token patching and a hard top-k, into a randomly-initialized concept dictionary — with the exact initialization and hyperparameters of A.1.
- **Information demonstrably flows through a 16-of-16k sparse channel** (C1's zero-ablation gap): behavior prediction is a real, self-supervised training signal for interpretability, the paper's central bet.
- **Concept death and its rescue by Eq 3 (C2)** reproduce readily — this dynamic is not an artifact of 8B scale.

### What it does NOT establish
- Anything about **absolute** interpretability quality: our auto-interp judge is zero-shot Haiku, not their finetuned simulator; our recall attributes are AG News topics, not SynthSys user attributes.
- The paper's **late-training divergence** (36M→72M) between aux and no-aux precision/recall — our budget ends before that regime. If your no-aux curves look fine, that is *consistent* with the paper, not a refutation.
- Downstream capabilities (jailbreak awareness etc.), which the paper shows **emerging with pretraining scale** (Figs 8, 9, 11) — far beyond this notebook.

### If you want to go further
1. **Scale the budget** (`CFG.token_budget = 8_000_000` on an A100 session) and watch whether the no-aux precision curve starts bending — the paper says it should.
2. **Train the TopK SAE baseline** (A.3) on the same activations and compare auto-interp scaling (Fig 4).
3. **Replace SynthSys with your own QA set** (Choi et al. 2025's recipe: dialogues with known latent attributes + consistency filtering) and finetune the decoder (§4) — this unlocks the §5 case studies.
4. Read next: **LatentQA** (Pan et al. 2024) — the no-bottleneck ancestor of the decoder; **Gao et al. 2024** — TopK SAEs and the aux-loss family; **Choi et al. 2025** — the SynthSys construction.